# 📱 Mobile Phone Recommendation System
**Content-Based Filtering using Cosine Similarity**

Dataset: Kaggle — Cellphones Recommendations (meirnizri)

---

## Task 1 — Load & Explore Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/cellphones_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# EDA — Brand distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Mobile Phone Dataset — Exploratory Data Analysis', fontsize=15, fontweight='bold')

# Brand count
brand_counts = df['brand'].value_counts()
axes[0,0].bar(brand_counts.index, brand_counts.values, color='steelblue', edgecolor='white')
axes[0,0].set_title('Phones per Brand')
axes[0,0].set_xlabel('Brand')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=45)

# Price distribution
axes[0,1].hist(df['price'], bins=10, color='coral', edgecolor='white')
axes[0,1].set_title('Price Distribution ($)')
axes[0,1].set_xlabel('Price (USD)')
axes[0,1].set_ylabel('Frequency')

# Battery vs Price
for brand in df['brand'].unique():
    sub = df[df['brand'] == brand]
    axes[1,0].scatter(sub['battery size'], sub['price'], label=brand, alpha=0.7)
axes[1,0].set_title('Battery Size vs Price')
axes[1,0].set_xlabel('Battery (mAh)')
axes[1,0].set_ylabel('Price ($)')
axes[1,0].legend(fontsize=7, ncol=2)

# Camera vs Price
axes[1,1].scatter(df['main camera'], df['price'], c='purple', alpha=0.6)
axes[1,1].set_title('Main Camera (MP) vs Price')
axes[1,1].set_xlabel('Camera (MP)')
axes[1,1].set_ylabel('Price ($)')

plt.tight_layout()
plt.savefig('../static/eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to static/eda_plots.png')

## Task 2 — Clean Dataset & Preprocess Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

df_clean = df.copy()

# --- Step 1: Parse release date ---
df_clean['release date'] = pd.to_datetime(df_clean['release date'], format='%d/%m/%Y')
df_clean['release_year'] = df_clean['release date'].dt.year
print('Parsed release dates.')

# --- Step 2: Encode OS ---
df_clean['os_encoded'] = (df_clean['operating system'] == 'iOS').astype(int)  # 1=iOS, 0=Android
print('Encoded OS: iOS=1, Android=0')

# --- Step 3: Define numeric feature columns ---
FEATURE_COLS = [
    'internal memory', 'RAM', 'performance',
    'main camera', 'selfie camera',
    'battery size', 'screen size', 'price'
]

# --- Step 4: Check and fill any nulls ---
for col in FEATURE_COLS:
    nulls = df_clean[col].isnull().sum()
    if nulls > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
        print(f'  Filled {nulls} nulls in {col} with median')
print(f'Null check complete. Total nulls: {df_clean[FEATURE_COLS].isnull().sum().sum()}')

# --- Step 5: Min-Max Normalization ---
scaler = MinMaxScaler()
df_clean[FEATURE_COLS] = scaler.fit_transform(df_clean[FEATURE_COLS])
print('\nMin-Max normalization applied to all feature columns.')

df_clean[FEATURE_COLS].head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df_clean[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap (Normalized)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## Task 3 — Build Content-Based Recommendation System

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

# Feature matrix — these columns represent each phone's normalized profile
# We exclude 'price' from similarity features (used separately for budget filter)
SIMILARITY_COLS = [
    'internal memory', 'RAM', 'performance',
    'main camera', 'selfie camera',
    'battery size', 'screen size'
]

feature_matrix = df_clean[SIMILARITY_COLS].values
print(f'Feature matrix shape: {feature_matrix.shape}')

# Pre-compute phone-to-phone cosine similarity matrix
similarity_matrix = cosine_similarity(feature_matrix)
print(f'Similarity matrix shape: {similarity_matrix.shape}')
print('\nSample similarity matrix (first 5x5):')
print(pd.DataFrame(similarity_matrix[:5, :5],
      index=df['model'][:5],
      columns=df['model'][:5]).round(3))

In [ ]:
def recommend_phones(budget_min, budget_max, os_pref,
                     camera_w, battery_w, ram_w,
                     performance_w, storage_w, selfie_w,
                     brand_pref=None, top_n=5):
    """
    Content-based phone recommender using cosine similarity.
    
    Parameters:
    -----------
    budget_min, budget_max : float  — price range in USD
    os_pref    : str  — 'iOS', 'Android', or 'Any'
    camera_w   : float (0–1) — main camera weight
    battery_w  : float (0–1) — battery size weight
    ram_w      : float (0–1) — RAM weight
    performance_w : float (0–1) — AnTuTu performance weight
    storage_w  : float (0–1) — internal memory weight
    selfie_w   : float (0–1) — selfie camera weight
    brand_pref : str  — optional preferred brand
    top_n      : int  — number of results
    """
    # Build user preference vector (same order as SIMILARITY_COLS)
    # [internal memory, RAM, performance, main camera, selfie camera, battery size, screen size]
    user_vec = np.array([
        storage_w, ram_w, performance_w,
        camera_w, selfie_w,
        battery_w, 0.5          # screen size — neutral weight
    ]).reshape(1, -1)
    
    # Filter candidates by budget
    candidates = df[df['price'].between(budget_min, budget_max)].copy()
    
    # Filter by OS
    if os_pref != 'Any':
        candidates = candidates[candidates['operating system'] == os_pref]
    
    if candidates.empty:
        return pd.DataFrame()
    
    # Get normalized feature vectors for candidates
    cand_features = df_clean.loc[candidates.index, SIMILARITY_COLS].values
    
    # Compute cosine similarity between user preference vector and each candidate
    scores = cosine_similarity(user_vec, cand_features)[0]
    
    # Apply brand bonus
    if brand_pref and brand_pref != 'Any':
        brand_bonus = (candidates['brand'] == brand_pref).values * 0.05
        scores = scores + brand_bonus
    
    candidates = candidates.copy()
    candidates['similarity_score'] = np.clip(scores, 0, 1)
    candidates['match_percent'] = (candidates['similarity_score'] * 100).round(1)
    
    result = candidates.sort_values('similarity_score', ascending=False).head(top_n)
    return result[['brand', 'model', 'operating system', 'internal memory', 'RAM',
                    'main camera', 'selfie camera', 'battery size',
                    'screen size', 'price', 'match_percent']]

# Quick test
result = recommend_phones(
    budget_min=300, budget_max=800, os_pref='Android',
    camera_w=0.9, battery_w=0.7, ram_w=0.6,
    performance_w=0.5, storage_w=0.4, selfie_w=0.3
)
print('=== Test recommendation: Budget Android with good camera ===')
result

## Task 4 — Evaluate Recommendations

In [ ]:
# 4 sample user personas for evaluation
test_users = [
    {
        'name': 'Budget Photographer',
        'params': dict(budget_min=100, budget_max=400, os_pref='Any',
                       camera_w=1.0, battery_w=0.5, ram_w=0.3,
                       performance_w=0.3, storage_w=0.4, selfie_w=0.8)
    },
    {
        'name': 'Power User (Samsung)',
        'params': dict(budget_min=400, budget_max=900, os_pref='Android',
                       camera_w=0.5, battery_w=0.9, ram_w=1.0,
                       performance_w=1.0, storage_w=0.8, selfie_w=0.3,
                       brand_pref='Samsung')
    },
    {
        'name': 'iPhone Loyalist',
        'params': dict(budget_min=700, budget_max=1500, os_pref='iOS',
                       camera_w=0.8, battery_w=0.6, ram_w=0.5,
                       performance_w=0.9, storage_w=0.5, selfie_w=0.6,
                       brand_pref='Apple')
    },
    {
        'name': 'Battery & Value Seeker',
        'params': dict(budget_min=100, budget_max=500, os_pref='Android',
                       camera_w=0.3, battery_w=1.0, ram_w=0.4,
                       performance_w=0.3, storage_w=0.5, selfie_w=0.2)
    },
]

avg_scores = []
for user in test_users:
    res = recommend_phones(**user['params'])
    avg = res['match_percent'].mean() if not res.empty else 0
    avg_scores.append(avg)
    print(f"\n{'='*55}")
    print(f"User: {user['name']}  |  Avg match: {avg:.1f}%")
    print('='*55)
    print(res[['brand','model','price','main camera','battery size','match_percent']].to_string(index=False))

In [ ]:
# Evaluation bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#4C9BE8', '#E85C4C', '#4CE88A', '#E8BC4C']
bars = ax.bar([u['name'] for u in test_users], avg_scores, color=colors, edgecolor='white', width=0.5)
for bar, score in zip(bars, avg_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{score:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Recommendation System — Avg Match Score by User Persona', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Cosine Similarity Match (%)')
ax.set_ylim(0, 105)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('../static/evaluation_chart.png', dpi=120, bbox_inches='tight')
plt.show()

## Save Model Artifacts with joblib

In [ ]:
os.makedirs('../models', exist_ok=True)

# 1. Save the MinMaxScaler
joblib.dump(scaler, '../models/scaler.pkl')
print('Saved: models/scaler.pkl')

# 2. Save the normalized feature matrix
joblib.dump(feature_matrix, '../models/feature_matrix.pkl')
print('Saved: models/feature_matrix.pkl')

# 3. Save the pre-computed similarity matrix
joblib.dump(similarity_matrix, '../models/similarity_matrix.pkl')
print('Saved: models/similarity_matrix.pkl')

# 4. Save original dataframe (raw prices needed for filtering)
joblib.dump(df, '../models/phones_df.pkl')
print('Saved: models/phones_df.pkl')

# 5. Save cleaned/normalized dataframe
joblib.dump(df_clean, '../models/phones_df_normalized.pkl')
print('Saved: models/phones_df_normalized.pkl')

# 6. Save feature column names
joblib.dump(SIMILARITY_COLS, '../models/feature_cols.pkl')
print('Saved: models/feature_cols.pkl')

print('\nAll model artifacts saved successfully!')

In [ ]:
# Verify saved models load correctly
s = joblib.load('../models/scaler.pkl')
fm = joblib.load('../models/feature_matrix.pkl')
sm = joblib.load('../models/similarity_matrix.pkl')
d = joblib.load('../models/phones_df.pkl')
print(f'Scaler: {type(s)}')
print(f'Feature matrix: {fm.shape}')
print(f'Similarity matrix: {sm.shape}')
print(f'Phones dataframe: {d.shape}')
print('\nAll models loaded and verified!')